# Byte I/O - Python

All 17 Python examples from [docs/io.md](https://platob.github.io/yggdryl/io/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
from yggdryl import IOBase

handle = IOBase.from_bytes()
handle.pwrite(0, b"symbol,price\n")
handle.pwrite(13, b"AAPL,1\n")
assert handle.size == 20

# Two reads at different offsets, in any order: there is no shared cursor.
assert handle.pread(13, 4) == b"AAPL"
assert handle.pread(0, 6) == b"symbol"

## Built from what you already hold

In [ ]:
import io

from yggdryl import IOBase

# An open file names its own location, so the handle addresses the path.
with open("quotes.json", "rb") as stream:
    handle = IOBase(stream)
assert handle.name == "quotes.json"

# A nameless stream holds only content, so the content is what is taken.
buffered = IOBase(io.BytesIO(b'{"symbol": "AAPL"}'))
buffered.media_type = "application/json"
assert buffered.read_text() == '{"symbol": "AAPL"}'

## Laziness

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp())

# Constructing touches nothing: no file is created, opened, or mapped.
handle = IOBase(root / "nested" / "lazy.csv")
assert not handle.exists()

# Reading something absent yields nothing rather than raising.
assert handle.size == 0
assert handle.read_bytes() == b""

# Writing creates the resource, and any parent it needs.
handle.write_text("symbol,price\n")
assert handle.is_file()
assert handle.read_text() == "symbol,price\n"

## Kinds

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase

folder = IOBase(pathlib.Path(tempfile.mkdtemp()))
assert folder.is_dir()
assert not folder.is_file()

# Nothing is there, so nothing has decided; a write settles it.
leaf = folder / "ticks.csv"
assert not leaf.exists()
leaf.write_text("symbol\n")
assert leaf.is_file()
assert not leaf.is_dir()

## Whole values

In [ ]:
from yggdryl import IOBase

handle = IOBase.from_bytes()
handle.write_bytes(b"symbol,price\n")

# `append` reports the offset the bytes landed at.
assert handle.append(b"AAPL,1\n") == 13
assert handle.pread(0, 6) == b"symbol"
# A range past the end yields what exists rather than raising.
assert handle.pread(100, 4) == b""
assert len(handle.read_bytes()) == 20

## Lines

In [ ]:
import gzip
import pathlib
import tempfile

from yggdryl import IOBase

target = pathlib.Path(tempfile.mkdtemp()) / "trades.jsonl.gz"
target.write_bytes(gzip.compress(b'{"id":1}\n{"id":2}\n'))

assert list(IOBase(target).read_lines()) == ['{"id":1}', '{"id":2}']

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase

target = pathlib.Path(tempfile.mkdtemp()) / "app.log"
target.write_text(
    "2024-02-01 10:00:00.000_000 [ee] [alpha] boom\n"
    "  at frame one\n"
    "2024-02-01 10:00:01.000_000 [ii] [beta] fine\n"
)

entries = list(IOBase(target).read_lines(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"))
assert len(entries) == 2
assert entries[0] == "2024-02-01 10:00:00.000_000 [ee] [alpha] boom\n  at frame one"

## What the bytes are

In [ ]:
from yggdryl import IOBase

# Nothing names an in-memory buffer, so its type comes from its bytes.
handle = IOBase.from_bytes(b'{"symbol":"AAPL"}')
assert str(handle.media_type.base) == "application/json"

# It is re-derived after the content changes.
handle.write_bytes(b"PAR1payload")
assert str(handle.media_type.base) == "application/vnd.apache.parquet"

## Open and close

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase

path = pathlib.Path(tempfile.mkdtemp()) / "trades.csv"

# `with` is the scoped pair: `__enter__` opens and `__exit__` closes.
with IOBase(path) as handle:
    handle.write_text("symbol,price\n")
    assert handle.is_open()

# Closing published the bytes at their exact length, which is what another
# reader needs; the handle stays usable and simply re-materializes.
assert path.stat().st_size == 13
assert IOBase(path).read_text() == "symbol,price\n"

In [ ]:
from yggdryl import IOBase

# Metadata-heavy work belongs inside the scope: the schema probe, the
# per-batch reads, and the size checks all reuse what `open` cached, and
# `close` releases it at a known point.
with IOBase("lake/trades.parquet") as handle:
    field = handle.read_arrow_field()
    for batch in handle.read_arrow_batch_reader():
        process(batch, field)

# Outside a scope the same calls still work - each one just fetches fresh,
# which is exactly right for a resource another writer may be changing.
latest = IOBase("lake/trades.parquet").read_arrow_field()

## Arrow batches

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

# A PyArrow schema is the schema; the binding imports it once at the boundary.
schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("symbol", pa.string()),
])
batch = pa.record_batch({"id": [1, 2], "symbol": ["AAPL", None]}, schema=schema)

# The handle's own media type picks the encoding; no format argument is passed.
handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.arrows")
options = handle.record_options()

# The write path takes a batch reader and nothing else.
handle.write_arrow_batch_reader(batch, options=options)
assert handle.read_arrow_field(options=options).name == "row"

# The read path returns one. Batches arrive one at a time, never as a vector.
rows = sum(part.num_rows for part in handle.read_arrow_batch_reader(options=options))
assert rows == 2

In [ ]:
import pathlib
import tempfile

import pytest

from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp())

# An absent resource holds no batches rather than failing to parse.
empty = IOBase(root / "absent.arrows")
assert empty.read_arrow_batch_reader().read_all().num_rows == 0

# An encoding this build does not implement is named rather than guessed.
csv = IOBase(root / "trades.csv")
with pytest.raises(ValueError, match="text/csv"):
    csv.record_options()

## Column pushdown

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

stored = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("symbol", pa.string(), nullable=False),
    pa.field("venue", pa.string(), nullable=False),
])
batch = pa.record_batch(
    {"id": [1, 2], "symbol": ["AAPL", "MSFT"], "venue": ["XNAS", "XNAS"]},
    schema=stored,
)

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.arrows")
handle.write_arrow_batch_reader(batch)

# One of the three columns, declared as this read's schema.
options = handle.record_options()
options.schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])

projected = handle.read_arrow_batch_reader(options=options)
assert projected.schema.names == ["id"]
assert projected.read_all().num_columns == 1

# The resource is unchanged: it still holds all three.
assert len(handle.read_arrow_field().data_type) == 3

# A column it does not hold cannot be projected out of it, so the encoding
# reads everything and the cast supplies that column as nulls.
options.schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("nowhere", pa.string()),
])
widened = handle.read_arrow_batch_reader(options=options)
assert widened.schema.names == ["id", "nowhere"]

## Appending and merging

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("symbol", pa.string()),
])
rows = lambda ids, symbols: pa.record_batch(
    {"id": ids, "symbol": symbols}, schema=schema
)

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.arrows")
options = handle.record_options()
options.schema = schema

# No match key: the resource is replaced.
handle.write_arrow_batch_reader(rows([1, 2], ["AAPL", "MSFT"]), options=options)

# Appending reads what is there, chains the new batches after it, and rewrites.
handle.append_arrow_batch_reader(rows([3], ["NVDA"]), options=options)
assert handle.read_arrow_batch_reader(options=options).read_all().num_rows == 3

# A match key merges: `2` is already stored and updates, `9` is new and appends.
merging = handle.record_options()
merging.schema = schema
merging.merge_by_names = ["id"]
handle.write_arrow_batch_reader(rows([2, 9], ["MSFT.O", "AMD"]), options=merging)
assert handle.read_arrow_batch_reader(options=options).read_all().num_rows == 4

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "orders.arrows")
handle.write_arrow(pa.table({"id": [1, 2], "symbol": ["AAPL", "MSFT"]}))

options = handle.record_options()
options.select_by_names = ["symbol"]
narrowed = handle.read_arrow(options=options).read_all()
assert narrowed.column_names == ["symbol"]

## Globbing and Hive partitions

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp()) / "lake"
for year in ("2024", "2025"):
    leaf = root / f"year={year}" / "month=01"
    leaf.mkdir(parents=True)
    (leaf / "part-0.parquet").write_bytes(b"parquet")

lake = IOBase(root)

assert len(lake.glob("year=2024/**/*.parquet")) == 1
assert len(lake.rglob("*.parquet")) == 2

selected = lake.children_where({"year": "2024"})
assert len(selected) == 1
assert selected[0].partitions == (("year", "2024"), ("month", "01"))

## Partition columns in the data

In [ ]:
import pathlib
import shutil
import tempfile

import pyarrow as pa

from yggdryl import IOBase, RecordOptions

root = pathlib.Path(tempfile.mkdtemp())
(root / "year=2024" / "month=01").mkdir(parents=True)

schema = pa.schema([
    pa.field("price", pa.int64(), nullable=False),
    pa.field("year", pa.int32(), nullable=False),
    pa.field("month", pa.string(), nullable=False),
])
batch = pa.record_batch(
    {"price": [10, 20], "year": [2024, 2024], "month": ["01", "01"]},
    schema=schema,
)

# The rows carry every column; the write drops the two the path spells out.
lake = IOBase(root)
options = RecordOptions("part.arrows")
options.schema = schema
lake.write_arrow_batch_reader(batch, options=options)

# Only `price` reached the leaf; the other two are the directory names.
leaf = lake / "year=2024" / "month=01" / "part-0.arrows"
assert len(leaf.read_arrow_field().data_type) == 1

# Reading the folder restores them with their declared types.
restored = lake.read_arrow_batch_reader(options=options).read_all()
assert restored.column_names == ["price", "year", "month"]
assert restored.schema.field("year").type == pa.int32()

shutil.rmtree(root)